# On-Device AI 실전 시험 대비 — 정답 및 핵심 개념 해설

신규 AVD 교육 커리큘럼 기반 23개 문항의 핵심 원리 및 빈칸별 출제 포인트 해설입니다.

In [ ]:
# [코드]
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


## 핵심 개념 1. Linear Quantization (FP → INT)

### 핵심 원리: 아핀(Affine) 선형 양자화 매핑
- **정의:** 32비트 부동소수점(`FP32`) 텐서를 `scale`(축척)과 `zero_point`(영점 이동) 두 개의 파라미터를 이용해 고정 비트 폭(`n`-bit)의 정수(`INT`) 공간으로 선형 사상합니다.
$$q = \text{clamp}\left(\text{round}\left(\frac{r}{S}\right) + Z,\; q_{\min},\; q_{\max}\right)$$

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`fp_tensor / scale`** *(Scaling)*:
   - 실수 값의 분포 폭을 정수 간격(step size = `scale`)으로 나누어 정수 격자에 대응되는 실수 좌표를 구합니다.
2. **`torch.round(scaled_tensor)`** *(Rounding)*:
   - 양자화 오차(quantization noise)를 최소화하기 위해 단순 버림(`int` 캐스팅)이 아닌 가장 가까운 정수로 **반올림**을 수행합니다. 이후 `.to(dtype)`으로 정수 타입으로 변환합니다.
3. **`rounded_tensor + zero_point`** *(Zero-point Shifting)*:
   - 실수 `0.0`이 양자화된 정수 공간에서 정확히 어느 정수값(`zero_point`)에 놓일지 평행 이동합니다.
   - **주의:** 덧셈 후 반드시 `get_quantized_range(bitwidth)` 범위 `[qmin, qmax]`로 `clamp_`해야 오버플로우(wrap-around)를 방지할 수 있습니다.

In [ ]:
# [코드]
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8) -> torch.Tensor:
    assert fp_tensor.dtype == torch.float
    scaled_tensor = ### fp_tensor / scale ### 작성 필요 ###  # 실수 텐서를 양자화 간격(scale)으로 나누어 정수 격자 스케일로 변환
    rounded_tensor = ### torch.round(scaled_tensor) ### 작성 필요 ###  # 양자화 오차 최소화를 위해 가장 가까운 정수로 반올림
    rounded_tensor = rounded_tensor.to(dtype)  # 양자화 오차 최소화를 위해 가장 가까운 정수로 반올림
    shifted_tensor = ### rounded_tensor + zero_point ### 작성 필요 ###  # 실수 0.0이 매핑되는 기준 오프셋(zero_point)만큼 평행 이동
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)  # 정수 비트 폭 유효 범위 [qmin, qmax]로 클램핑해 오버플로우 방지
    return quantized_tensor

## 핵심 개념 2. Quantization Scale & Zero Point 계산

### 핵심 원리: 동적 범위(Dynamic Range) 정렬
- **정의:** 비대칭 양자화(Asymmetric Quantization)에서는 텐서의 실제 최솟값 `fp_min`과 최댓값 `fp_max`를 양자화 정수 범위 `[qmin, qmax]` 양 끝단에 정확히 맞춥니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`(fp_max - fp_min) / (quantized_max - quantized_min)`** *(Scale 계산)*:
   - 실수 구간의 총 길이(`fp_max - fp_min`)를 정수 구간 칸 수(`2^bitwidth - 1`)로 나누어 **정수 1칸이 표현하는 실수 크기(`scale`)**를 산출합니다.
2. **`round(quantized_min - fp_min / scale)`** *(Zero Point 계산)*:
   - 역양자화 식 $r_{\min} = (q_{\min} - Z) \cdot S$ 를 $Z$에 대해 정리하면 $Z = q_{\min} - r_{\min} / S$ 가 됩니다.
   - `zero_point`는 정수여야 실수 `0.0`을 오차 없이 표현할 수 있으므로 `round()`를 취하고, 범위 이탈 시 `[qmin, qmax]`로 클리핑합니다.

In [ ]:
# [코드]
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    quantized_min, quantized_max = get_quantized_range(bitwidth)  # 비트 폭에 따른 정수 표현 최솟값/최댓값 조회 (예: 8bit signed -> -128, 127)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    scale = ### (fp_max - fp_min) / (quantized_max - quantized_min) ### 작성 필요 ###  # 실수 구간 폭(fp_max - fp_min)을 정수 구간 칸 수로 나누어 1스텝 크기 산출
    zero_point = ### round(quantized_min - fp_min / scale) ### 작성 필요 ###  # 실수 최솟값이 quantized_min에 대응되도록 정수 영점 계산

    if zero_point < quantized_min:
        zero_point = quantized_min  # 실수 최솟값이 quantized_min에 대응되도록 정수 영점 계산
    elif zero_point > quantized_max:
        zero_point = quantized_max  # 실수 최솟값이 quantized_min에 대응되도록 정수 영점 계산
    else:
        zero_point = round(zero_point)  # 실수 최솟값이 quantized_min에 대응되도록 정수 영점 계산
    return scale, int(zero_point)

## 핵심 개념 3. Per-Channel Weight Quantization

### 핵심 원리: 채널별(Per-Channel) 대칭 양자화
- **정의:** CNN/Linear 가중치 텐서는 출력 채널(`dim=0`)마다 값의 크기 편차가 큽니다. 텐서 전체에 하나의 `scale`을 쓰는 Per-Tensor 방식보다 출력 채널마다 독립적인 `scale`을 구하면 양자화 정밀도가 크게 향상됩니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`tensor.select(dim_output_channels, oc)`** *(채널 슬라이싱)*:
   - `dim=0` 축을 따라 `oc`번째 출력 채널 필터 텐서만 추출하여 해당 채널 전용 스케일 `_scale`을 계산합니다.
2. **`scale.view(scale_shape)`** *(브로드캐스팅 형상 정렬)*:
   - 1D 벡터인 `scale` `(C_out,)`을 원본 가중치 `tensor`의 차원에 맞춰 `(C_out, 1, ...)` 형태로 reshape해야 `linear_quantize` 내부의 `fp_tensor / scale` 나눗셈이 채널 축으로 정확히 브로드캐스팅됩니다.

In [ ]:
# [코드]
def linear_quantize_weight_per_channel(tensor, bitwidth):  # 대칭 양자화(zero_point=0) 수행
    dim_output_channels = 0
    num_output_channels = tensor.shape[dim_output_channels]
    scale = torch.zeros(num_output_channels, device=tensor.device)

    for oc in range(num_output_channels):  # 출력 채널 축(dim=0)을 순회하며 채널마다 별도 동적 범위 계산
        _subtensor = ### tensor.select(dim_output_channels, oc) ### 작성 필요 ###  # oc번째 출력 채널 필터 슬라이스만 추출
        _scale = get_quantization_scale_for_weight(_subtensor, bitwidth)
        scale[oc] = _scale

    scale_shape = [1] * tensor.dim()
    scale_shape[dim_output_channels] = -1
    scale = ### scale.view(scale_shape) ### 작성 필요 ###
    quantized_tensor = linear_quantize(tensor, bitwidth, scale, zero_point=0)  # 대칭 양자화(zero_point=0) 수행
    return quantized_tensor, scale, 0

## 핵심 개념 4. Quantized Linear: Output 재스케일 & Zero Point 보정

### 핵심 원리: 정수 연산(INT32 Accumulator)의 Requantization
- **정의:** INT8 입력과 INT8 가중치의 행렬 곱(`F.linear`) 결과는 오버플로우를 막기 위해 32비트 정수(`INT32`)로 누적됩니다. 이를 다음 레이어의 INT8 입력으로 넘기기 위해 스케일 비율 보정(Requantization)이 필수입니다.
$$q_{\text{out}} = \text{clamp}\left(\text{round}\left(\frac{S_{\text{in}} \cdot S_W}{S_{\text{out}}} \cdot \text{acc}_{\text{int32}} + Z_{\text{out}}\right),\; q_{\min},\; q_{\max}\right)$$

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`weight_scale.view(1, -1)`** *(차원 정렬)*:
   - Linear 출력 텐서 `(B, out_features)`의 마지막 채널 축과 곱해질 수 있도록 Per-channel `weight_scale`을 `(1, out_features)` 형태로 맞춥니다.
2. **`output * (input_scale * weight_scale_reshaped / output_scale) + output_zero_point`** *(재스케일링 + 영점 복원)*:
   - 입력 스케일 × 가중치 스케일로 역양자화한 값을 다시 `output_scale`로 나누어 출력 양자화 격자에 맞추고, `output_zero_point`를 더해 양자화된 정수 오프셋을 복원합니다.

In [ ]:
# [코드]
def quantized_linear(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale):
    output = torch.nn.functional.linear(input.to(torch.int32), weight.to(torch.int32), bias)  # 오버플로우 방지를 위해 INT8 입력/가중치를 INT32로 업캐스팅해 행렬 곱 누적
    weight_scale_reshaped = ### weight_scale.view(1, -1) ### 작성 필요 ###  # 출력 채널 축(dim=1)에 맞도록 (1, out_features) 형태로 정렬
    output = ### output * (input_scale * weight_scale_reshaped / output_scale) + output_zero_point ### 작성 필요 ###  # INT32 누산기에 실효 스케일 비율을 곱하고 output_zero_point 보정
    qmin, qmax = get_quantized_range(feature_bitwidth)
    output = output.round().clamp(qmin, qmax).to(torch.int8)  # 반올림 후 출력 비트 폭 범위 [qmin, qmax]로 클램핑하여 INT8 변환
    return output

## 핵심 개념 5. Quantized Conv2d: Output 재스케일 & Zero Point 보정

### 핵심 원리: 4D 합성곱 피처맵 Requantization
- **정의:** 2D 합성곱 출력 텐서는 `(N, C_out, H, W)`의 4차원 구조를 갖습니다. 따라서 채널별 `bias` 및 `weight_scale`은 2차원 Linear와 달리 `(1, C_out, 1, 1)` 형상으로 브로드캐스팅되어야 합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`bias.view(1, -1, 1, 1)`**:
   - INT32 누적 결과(`(N, C_out, H, W)`)의 `C_out` 축(`dim=1`)에 편향을 더하기 위해 4D 형태로 확장합니다.
2. **`output * (input_scale * weight_scale.view(1, -1, 1, 1) / output_scale) + output_zero_point`**:
   - Per-channel 가중치 스케일 역시 `(1, C_out, 1, 1)`로 맞추어 실효 스케일 비율을 곱하고, 출력 제로 포인트를 더한 뒤 8비트 정수 범위로 클램핑합니다.

In [ ]:
# [코드]
def quantized_conv2d(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale,
                     stride, padding, dilation, groups):
    input = torch.nn.functional.pad(input, padding, 'constant', input_zero_point)  # 합성곱 패딩 영역을 일반 0이 아닌 input_zero_point(실수 0.0)로 채움
    output = torch.nn.functional.conv2d(input.to(torch.int32), weight.to(torch.int32), None, stride, 0, dilation, groups)  # INT32 정밀도로 합성곱 누적 연산 수행 (bias는 나중에 별도 가산)
    if bias is not None:
        output = output + ### bias.view(1, -1, 1, 1) ### 작성 필요 ###  # 4D 피처맵 (N, C, H, W)의 채널 축에 맞게 bias 브로드캐스팅
    output = ### output * (input_scale * weight_scale.view(1, -1, 1, 1) / output_scale) + output_zero_point ### 작성 필요 ###  # (input_scale * weight_scale / output_scale) 재스케일 및 output_zero_point 보정
    qmin, qmax = get_quantized_range(feature_bitwidth)
    output = output.round().clamp(qmin, qmax).to(torch.int8)
    return output

## 핵심 개념 6. K-means Quantization & Codebook 디코딩

### 핵심 원리: 가중치 공유(Weight Sharing) 기반 비균등 양자화
- **정의:** K-means 군집화를 통해 유사한 가중치 값들을 `2^bitwidth`개의 대표 중심값(Centroid)으로 묶고, 원래 텐서에는 중심값 인덱스(`labels`)만 저장하여 모델을 극적으로 압축합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`2 ** bitwidth`** *(클러스터 수)*:
   - 인덱스를 `b`비트로 표현할 수 있는 고유 상태 개수는 $2^b$개입니다. (예: 4-bit → 16개 중심값)
2. **`codebook.centroids[codebook.labels]`** *(코드북 디코딩)*:
   - 추론 시에는 정수 인덱스 텐서 `codebook.labels`로 `codebook.centroids` 배열을 룩업(인덱싱)하여 실제 가중치와 같은 크기의 근사 텐서를 즉시 복원합니다.

In [ ]:
# [코드]
def k_means_quantize(fp32_tensor: torch.Tensor, bitwidth=4, codebook=None):
    if codebook is None:
        n_clusters = ### 2 ** bitwidth ### 작성 필요 ###  # bitwidth 비트로 표현 가능한 총 대표 중심값 개수 (2^bitwidth) 설정
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)  # 1D 가중치 원소들을 n_clusters 개수로 유클리디안 K-means 군집화
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)
    quantized_tensor = ### codebook.centroids[codebook.labels] ### 작성 필요 ###  # 정수 레이블 인덱스로 centroids 룩업 테이블을 조회해 근사 텐서 복원
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))  # 원본 가중치 메모리를 복원된 코드북 가중치로 인플레이스 교체
    return codebook

## 핵심 개념 7. K-means Codebook 업데이트 (QAT)

### 핵심 원리: Quantization-Aware Training(QAT)에서의 Centroid 재계산
- **정의:** 역전파를 통해 연속적인 `fp32_tensor` 가중치가 학습으로 갱신되면, 기존 클러스터 배정(`labels`)은 유지한 채 각 클러스터에 속한 가중치들의 **새로운 평균값**으로 코드북 중심점(`centroids`)을 동기화합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`torch.mean(fp32_tensor[codebook.labels == k])`** *(클러스터 평균 산출)*:
   - 1차원으로 펼친 `fp32_tensor`에서 불리언 마스크 `codebook.labels == k`로 `k`번째 클러스터에 속한 원소들만 필터링한 뒤 평균을 계산하여 중심점을 최신화합니다.

In [ ]:
# [코드]
def update_codebook(fp32_tensor: torch.Tensor, codebook: Codebook):
    n_clusters = codebook.centroids.numel()  # 코드북 내 중심값(centroid) 총 개수
    fp32_tensor = fp32_tensor.view(-1)  # 라벨 텐서와 길이를 맞추기 위해 1차원으로 평탄화
    for k in range(n_clusters):
        codebook.centroids[k] = ### torch.mean(fp32_tensor[codebook.labels == k]) ### 작성 필요 ###  # k번 클러스터에 속한 학습된 가중치들의 새로운 평균으로 중심점 갱신

## 핵심 개념 8. AWQ: Salient Channel Scaling (상위 1% 중요 채널 스케일링)

### 핵심 원리: Activation-aware Weight Quantization (AWQ)
- **정의:** LLM 가중치의 모든 채널이 똑같이 중요한 것이 아니라, 입력 활성화(`input_feat`) 크기가 큰 **상위 약 1%의 중요 채널(Salient Channels)**이 출력 오차를 지배합니다.
- 이 중요 채널들의 가중치만 양자화 전에 `scale_factor`를 곱해 키운 뒤 양자화하고, 이후 다시 `scale_factor`로 나누어 정밀도를 보존합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`torch.topk(importance, num_samples)[1]`** *(상위 1% 채널 인덱스 추출)*:
   - `torch.topk()`는 `(values, indices)` 튜플을 반환합니다. 여기서 인덱스(`[1]`)를 꺼내야 가중치 행렬의 열(`[:, outlier_mask]`)을 슬라이싱할 수 있습니다.
2. **`m.weight.data[:, outlier_mask] /= scale_factor`** *(스케일 복원)*:
   - 양자화 전 키웠던(`*= scale_factor`) 중요 채널들을 양자화 직후 동일한 비율로 나누어야 원래의 수학적 선형 변환($W X$) 등가성이 유지됩니다.

In [ ]:
# [코드]
@torch.no_grad()
def pseudo_quantize_model_weight_scaleup(model, w_bit, q_group_size, input_feat, scale_factor):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            importance = sum(input_feat[n]).float()  # 입력 활성화 크기를 채널별 중요도로 합산
            num_samples = int(len(importance) * 0.01)  # 전체 채널 중 상위 1%에 해당하는 채널 수 산출
            outlier_mask = ### torch.topk(importance, num_samples)[1] ### 작성 필요 ###  # 중요도가 가장 높은 상위 1% 채널의 인덱스 추출 ([1]은 indices)
            assert outlier_mask.dim() == 1
            m.weight.data[:, outlier_mask] *= scale_factor  # 양자화 전 중요 채널 가중치를 키워 정밀도 보호
            m.weight.data = pseudo_quantize_tensor(m.weight.data, n_bit=w_bit, q_group_size=q_group_size)
            ### m.weight.data[:, outlier_mask] /= scale_factor ### 작성 필요 ###

## 핵심 개념 9. SmoothQuant: LayerNorm ↔ Linear 난이도 전이

### 핵심 원리: Smoothing (난이도 전이)
- **정의:** LLM에서는 활성화(`X`)에 수십~수백 배에 달하는 이상치(Outlier)가 특정 채널에 몰려 있어 W8A8 양자화가 어렵습니다. SmoothQuant는 수학적 항등식 $Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$ 를 이용해 **활성화의 이상치를 가중치 쪽으로 전이**시킵니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`ln.weight.div_(scale)` & `ln.bias.div_(scale)`** *(활성화 출력 축소)*:
   - LayerNorm의 아핀 파라미터를 `scale`로 나누어 출력 활성화 크기를 평탄하게 줄여줍니다 (`div_`는 인플레이스 연산).
2. **`fc.weight.mul_(scale)`** *(후속 가중치 보상)*:
   - LayerNorm 다음 단에 연결된 모든 Linear 레이어의 가중치 입력 채널에 동일한 `scale`을 곱해 전체 순전파 출력이 변하지 않도록 상쇄합니다.

In [ ]:
# [코드]
@torch.no_grad()
def smooth_ln_fcs_by_scale(ln, fcs, scale):
    if not isinstance(fcs, list):
        fcs = [fcs]
    assert isinstance(ln, nn.LayerNorm)
    for fc in fcs:
        assert isinstance(fc, nn.Linear)

    ### ln.weight.div_(scale) ### 작성 필요 ###  # LayerNorm 가중치를 scale로 나누어 활성화 이상치 축소
    ### ln.bias.div_(scale) ### 작성 필요 ###  # LayerNorm 편향 역시 scale로 나누어 분포 유지

    for fc in fcs:
        ### fc.weight.mul_(scale) ### 작성 필요 ###  # 후속 Linear 입력 가중치에 scale을 곱해 출력 등가성 보상

## 핵심 개념 10. SmoothQuant: Alpha 기반 Scale Factor 계산

### 핵심 원리: 이동 강도 하이퍼파라미터 $\alpha$
- **정의:** 채널별 최적 smoothing factor $s_j$는 활성화의 최대 크기(`act_scales`)와 가중치의 최대 크기(`weight_scales`)를 지수 $\alpha$로 조화롭게 조합하여 결정합니다.
$$s_j = \frac{\max(|X_j|)^\alpha}{\max(|W_j|)^{1-\alpha}}$$

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`act_scales.pow(alpha) / weight_scales.pow(1 - alpha)`**:
   - `alpha=0.5`이면 활성화와 가중치가 양자화 난이도를 절반씩 부담하며, `alpha`가 1에 가까울수록 활성화를 더 강하게 누르고 가중치로 더 많은 난이도를 넘깁니다.

In [ ]:
# [코드]
@torch.no_grad()
def smooth_ln_fcs(ln, fcs, act_scales, alpha=0.5):
    if not isinstance(fcs, list):
        fcs = [fcs]
    device, dtype = fcs[0].weight.device, fcs[0].weight.dtype
    act_scales = act_scales.to(device=device, dtype=dtype)  # alpha 비율로 활성화(act^alpha)와 가중치(w^(1-alpha)) 간 양자화 난이도 분배
    weight_scales = torch.cat(  # 연결된 FC 레이어들의 입력 채널별 최대 가중치 크기 추출
        [fc.weight.abs().max(dim=0, keepdim=True)[0] for fc in fcs], dim=0
    )
    weight_scales = weight_scales.max(dim=0)[0].clamp(min=1e-5)  # alpha 비율로 활성화(act^alpha)와 가중치(w^(1-alpha)) 간 양자화 난이도 분배

    scales = ### act_scales.pow(alpha) / weight_scales.pow(1 - alpha) ### 작성 필요 ###  # alpha 비율로 활성화(act^alpha)와 가중치(w^(1-alpha)) 간 양자화 난이도 분배
    scales = scales.clamp(min=1e-5).to(device).to(dtype)  # alpha 비율로 활성화(act^alpha)와 가중치(w^(1-alpha)) 간 양자화 난이도 분배

    ln.weight.div_(scales)
    ln.bias.div_(scales)
    for fc in fcs:
        fc.weight.mul_(scales.view(1, -1))

## 핵심 개념 11. Rotation-based Quantization: 직교 행렬 가중치 회전

### 핵심 원리: QuaRot / SpinQuant 가중치 공간 회전
- **정의:** 직교 행렬(Orthogonal Matrix, $R_1 R_1^T = I$)을 이용해 활성화와 가중치를 회전시키면 이상치 에너지가 모든 차원으로 균일하게 분산(Incoherence Processing)되어 극저비트(4-bit 이하) 양자화 성능이 비약적으로 향상됩니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **Embedding: `W_ @ R1`**
   - 임베딩 행렬 형상은 `(vocab_size, hidden_dim)`이므로, 우측에 `R1`을 곱해 출력 은닉 차원을 회전 공간으로 보냅니다.
2. **출력 프로젝션 (`o_proj`, `down_proj`): `R1.T @ W_`**
   - PyTorch Linear의 가중치는 `(out_features, in_features)`로 전치 저장됩니다. 잔차 스트림으로 복귀하는 출력 프로젝션은 출력 차원에 역회전 $R_1^T$를 좌측에 곱해 공간을 되돌립니다.
3. **입력 프로젝션 (Q, K, V, Gate, Up): `W_ @ R1`**
   - 회전된 입력 활성화 $X R_1$과 곱해지므로, 가중치의 입력 축(`in_features`)에 `R1`을 우측 곱셈하여 $X R_1 \cdot R_1^T W^T = X W^T$로 상쇄시킵니다.

In [ ]:
# [코드]
def rotate_model_weight(model, R1):
    for n, m in model.named_modules():
        if isinstance(m, nn.Embedding):  # 임베딩 테이블 (vocab, hidden) 출력 차원에 직교 행렬 R1 우측 곱셈
            W_ = m.weight.data
            m.weight.data = ### W_ @ R1 ### 작성 필요 ###  # Q/K/V/Gate/Up 입력 축은 회전된 활성화(X@R1)와 상쇄되도록 R1 우측 곱셈
        if isinstance(m, nn.Linear):
            if "o_proj" in n or "down_proj" in n:
                W_ = m.weight.data
                m.weight.data = ### R1.T @ W_ ### 작성 필요 ###  # o_proj/down_proj 출력 축을 원래 잔차 공간으로 되돌리기 위해 역행렬 R1.T 좌측 곱셈
            else:
                W_ = m.weight.data
                m.weight.data = ### W_ @ R1 ### 작성 필요 ###  # Q/K/V/Gate/Up 입력 축은 회전된 활성화(X@R1)와 상쇄되도록 R1 우측 곱셈

## 핵심 개념 12. Knowledge Distillation (Soft Targets KL Loss)

### 핵심 원리: Temperature Scaling 기반 Response-based KD
- **정의:** Teacher 모델의 로짓을 온도 $T > 1$ 로 나누어 softmax를 취하면 클래스 간 상대적 유사성(Dark Knowledge)이 부드럽게 드러납니다. Student가 이를 모방하도록 학습합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`torch.no_grad()`**:
   - Teacher 모델은 고정(Frozen)되어 학습하지 않으므로 메모리와 연산 절약을 위해 기울기 그래프를 기록하지 않습니다.
2. **`nn.functional.softmax(..., dim=-1)`** *(온도 $T$ 적용)*:
   - 로짓을 `T`로 나눈 뒤 마지막 클래스 차원(`dim=-1`)에 소프트맥스를 적용하여 완화된 확률 분포 `soft_targets`, `student_prob`를 생성합니다.
3. **`(T ** 2)`** *(Gradient 스케일 보정)*:
   - 로짓을 `T`로 나누면 역전파 시 기울기 크기가 $1/T^2$로 줄어듭니다. 따라서 Hard Label 손실(`ce_loss`)과 균형을 맞추기 위해 반드시 $T^2$을 곱해 보정합니다.

In [ ]:
# [코드]
def kd_step(teacher, student, inputs, labels, T, soft_target_loss_weight, ce_loss_weight, ce_loss):
    with ### torch.no_grad() ### 작성 필요 ###:  # 고정된 Teacher 모델 추론 시 역전파 그래프 기록 생략
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)

    soft_targets = ### nn.functional.softmax(teacher_logits / T, dim=-1) ### 작성 필요 ###  # Teacher 로짓을 온도 T로 나누어 클래스 간 유사도(Dark Knowledge) 완화
    student_prob = ### nn.functional.softmax(student_logits / T, dim=-1) ### 작성 필요 ###  # Student 로짓 역시 동일 온도 T로 소프트맥스 확률 분포 생성

    soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - student_prob.log()), dim=1).mean() * ### (T ** 2) ### 작성 필요 ###  # 온도 T 나눗셈으로 1/T^2 만큼 작아진 gradient 크기를 보정
    label_loss = ce_loss(student_logits, labels)
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss  # Soft Target KL 손실과 Hard Label CE 손실을 가중합
    return loss

## 핵심 개념 13. Cosine Similarity Loss 기반 Knowledge Distillation

### 핵심 원리: Feature Representation 방향 정렬
- **정의:** 최종 예측 분포뿐 아니라 중간 은닉 표현 벡터의 방향(Cosine Similarity)이 Teacher와 일치하도록 학습하여 깊은 특징 표현력을 전수합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(inputs.device))`**:
   - PyTorch `nn.CosineEmbeddingLoss(x1, x2, y)`는 **`y = +1`일 때 $1 - \cos(x_1, x_2)$를 반환**하여 두 벡터의 방향이 같아지도록 학습시킵니다.
   - **주의:** `target` 텐서는 반드시 배치 크기(`inputs.size(0)`)와 동일한 길이의 `1` 벡터여야 하며 같은 `device`에 위치해야 합니다.

In [ ]:
# [코드]
def cosine_kd_step(teacher, student, inputs, labels, hidden_rep_loss_weight, ce_loss_weight, ce_loss, cosine_loss):
    with torch.no_grad():
        _, teacher_hidden_representation = teacher(inputs)  # Teacher 모델의 중간 은닉 표현 벡터 추출 (분류 로짓은 무시)
    student_logits, student_hidden_representation = student(inputs)

    hidden_rep_loss = ### cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(inputs.device)) ### 작성 필요 ###  # target=+1 설정으로 두 벡터의 코사인 유사도가 1이 되도록 방향 정렬
    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 핵심 개념 14. Intermediate Regressor 기반 MSE Feature Map Distillation

### 핵심 원리: FitNets (Hint-based Feature Map Distillation)
- **정의:** 크기가 작은 Student 모델은 Teacher와 중간 피처맵의 채널/공간 차원이 다를 수 있습니다. Student에 `1x1 Conv` Regressor를 붙여 차원을 맞춘 뒤 요소별 MSE(평균 제곱 오차)를 최소화합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`mse_loss(regressor_feature_map, teacher_feature_map)`**:
   - Teacher 피처맵(`teacher_feature_map`)과 Regressor를 통과해 차원이 정렬된 Student 피처맵(`regressor_feature_map`) 간의 유클리디안 거리 제곱 오차를 계산합니다.

In [ ]:
# [코드]
def mse_kd_step(teacher, student, inputs, labels, feature_map_weight, ce_loss_weight, ce_loss, mse_loss):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)  # Teacher 모델의 중간 공간 피처맵 추출
    student_logits, regressor_feature_map = student(inputs)

    hidden_rep_loss = ### mse_loss(regressor_feature_map, teacher_feature_map) ### 작성 필요 ###  # Regressor로 차원을 맞춘 Student 피처맵과 Teacher 피처맵 간 MSE 오차 계산
    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 핵심 개념 15. KV Cache 메모리 사용량(Bytes) 계산

### 핵심 원리: Auto-regressive LLM의 KV Cache 메모리 병목
- **정의:** LLM은 매 토큰 생성 시 과거 모든 토큰의 Key, Value 벡터를 캐싱하여 중복 계산을 피합니다. 문맥 길이(`seq_len`)가 길어질수록 메모리 사용량이 선형으로 폭증하여 온디바이스 배포의 최대 병목이 됩니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`2 * batch_size * n_layers * n_kv_heads * seq_len * head_dim * dtype_bytes`**:
   - **계수 `2`**: Key 텐서와 Value 텐서 2종류를 모두 저장해야 합니다.
   - **`n_kv_heads`**: GQA(Grouped-Query Attention) 모델에서는 `num_attention_heads`가 아니라 실제 캐싱되는 KV 헤드 수(`num_key_value_heads`)를 곱해야 정확합니다.
   - **`dtype_bytes`**: FP16/BF16이면 2바이트, FP32면 4바이트를 곱합니다.

In [ ]:
# [코드]
def kv_cache_bytes(config, seq_len, batch_size=1, dtype_bytes=2):
    n_layers = config.num_hidden_layers
    n_kv_heads = config.num_key_value_heads  # GQA 모델에서는 Query 헤드가 아닌 실제 캐싱되는 KV 헤드 수를 사용
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)

    total_bytes = ### 2 * batch_size * n_layers * n_kv_heads * seq_len * head_dim * dtype_bytes ### 작성 필요 ###  # 2(Key+Value) * 배치 * 레이어 * KV헤드 * 길이 * 헤드차원 * 원소바이트
    return total_bytes

## 핵심 개념 16. H2O (Heavy-Hitter Oracle) Attention 누적 스코어링

### 핵심 원리: Heavy-Hitter + Recent Window Eviction 정책
- **정의:** H2O(Heavy-Hitter Oracle)는 과거 어텐션 가중치의 누적 합이 높은 소수의 토큰(Heavy Hitters)과 지역적 문맥을 담당하는 최근 토큰(`recent_size`)만을 캐시에 남기고 나머지를 제거(Eviction)합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`attn_kv.sum(dim=-2)`** *(쿼리 축 누적 합산)*:
   - 어텐션 텐서 shape `[B, H, Q, K]`에서 쿼리 축(`dim=-2`) 전체에 대해 합산하면, 각 키 토큰(`K`)이 얼마나 많은 총 어텐션을 받았는지를 나타내는 `[B, H, K]` 점수 벡터를 얻습니다.
2. **`scores[..., -recent_size:] = float("inf")`** *(최근 토큰 영구 보존)*:
   - 상위 `budget`개를 `topk`로 고를 때 최근 윈도우 토큰들이 절대 탈락하지 않도록 음이 아닌 무한대(`inf`)를 할당합니다.

In [ ]:
# [코드]
def h2o_score(attn_kv, layer, budget):
    recent_size = budget // 2  # 전체 캐시 예산(budget) 중 절반을 최근 지역 문맥 윈도우로 예약
    scores = ### attn_kv.sum(dim=-2) ### 작성 필요 ###  # 모든 쿼리(dim=-2)가 각 키 토큰에 준 어텐션 가중치를 누적 합산
    ### scores[..., -recent_size:] = float("inf") ### 작성 필요 ###
    return scores

## 핵심 개념 17. SnapKV: Observation Window & Max Pooling 스코어링

### 핵심 원리: SnapKV (프롬프트 말미 관측 + 클러스터링 보존)
- **정의:** H2O와 달리 프롬프트 전체 쿼리가 아닌 **마지막 관측 윈도우(`window`) 내 쿼리들의 어텐션**만 보고 중요 KV를 선별합니다(생성 단계와 유사한 어텐션 패턴). 또한 1D Max Pooling으로 중요한 토큰 주변의 문맥까지 통째로 살려 정보 손실을 방지합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`attn_kv[:, :, -w:, :].sum(dim=2)`** *(관측 윈도우 어텐션 집계)*:
   - 쿼리 축(`dim=2`)에서 뒤쪽 `w`개(`-w:`)만 슬라이싱해 합산하므로 앞쪽 편향 없이 생성 시점의 주의 집중도를 정확히 반영합니다.
2. **`scores[:, :, start:end].max(dim=-1).values`** *(인접 커널 Max Pooling)*:
   - 위치 `j` 주변 `[j-half, j+half]` 구간의 최대 점수를 할당하여 단일 토큰이 아닌 구절/문맥 단위로 점수가 보존되게 합니다. (`max`는 `(values, indices)`를 반환하므로 `.values` 필수!)
3. **`scores[:, :, -w:] = float("inf")`** *(관측 윈도우 영구 보존)*:
   - 관측 윈도우 자체(`-w:`)는 다음 토큰 생성에 직결되므로 항상 `inf` 점수로 유지합니다.

In [ ]:
# [코드]
def snapkv_score(attn_kv, layer, budget, window=32, kernel=7):
    B, H, Q, K = attn_kv.shape
    w = min(window, Q)  # 프롬프트 길이가 window보다 짧을 경우를 대비한 관측 윈도우 크기 안전 설정

    scores = ### attn_kv[:, :, -w:, :].sum(dim=2) ### 작성 필요 ###  # 마지막 w개 관측 쿼리가 각 키 토큰에 준 어텐션만 합산해 생성 패턴 모사

    half = kernel // 2
    pooled_scores = scores.clone()
    for j in range(K):
        start = max(0, j - half)
        end = min(K, j + half + 1)
        pooled_scores[:, :, j] = ### scores[:, :, start:end].max(dim=-1).values ### 작성 필요 ###  # 위치 j 주변 커널 구간 내 최댓값 풀링으로 문맥 청크 단위 보존

    scores = pooled_scores
    ### scores[:, :, -w:] = float("inf") ### 작성 필요 ###
    return scores

## 핵심 개념 18. LLM Magnitude-based Pruning

### 핵심 원리: 가중치 크기 기반 비구조적 프루닝 (Unstructured Pruning)
- **정의:** 0에 가까운 작은 가중치들은 모델 출력에 미치는 영향이 미미하다고 가정하고, 절댓값 하위 `sparsity` 비율만큼 0으로 마스킹하여 희소 행렬(Sparse Matrix)을 만듭니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`round(num_elements * sparsity)`**: 제거할 총 원소 개수(`k`번째 작은 값을 찾기 위한 인덱스)를 정수로 산출합니다.
2. **`torch.abs(W)`**: 부호에 관계없이 크기(magnitude)로 중요도를 판단합니다.
3. **`torch.kthvalue(importance.flatten(), num_zeros)[0]`**: 1D로 펼친 중요도에서 `num_zeros`번째로 작은 경계값(threshold)을 찾습니다.
4. **`importance > threshold`**: 경계값보다 큰 원소만 `True`(1), 이하인 원소는 `False`(0)가 되도록 마스크를 생성해 `W.mul_(mask)`로 제거합니다.

In [ ]:
# [코드]
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            num_elements = W.numel()
            num_zeros = ### round(num_elements * sparsity) ### 작성 필요 ###  # 전체 파라미터 수 * 희소도(sparsity) 반올림으로 제거 개수 결정
            importance = ### torch.abs(W) ### 작성 필요 ###  # 부호 무관하게 가중치 절댓값 크기를 중요도로 평가
            threshold = ### torch.kthvalue(importance.flatten(), num_zeros)[0] ### 작성 필요 ###  # 1D 평탄화 중요도에서 num_zeros번째로 작은 경계값 산출
            mask = ### importance > threshold ### 작성 필요 ###  # 경계값 초과 원소만 True(1)로 남기고 나머지는 0으로 마스킹
            W.mul_(mask)

## 핵심 개념 19. Wanda용 Activation L2 Norm 누적

### 핵심 원리: Wanda의 입력 활성화 통계량($\|X_j\|_2$) 수집
- **정의:** 가중치 크기만 보는 Magnitude Pruning은 큰 활성화 피처와 곱해지는 중요한 가중치를 오인 제거할 수 있습니다. Wanda는 보정 데이터(Calibration Set)를 통과시키며 각 입력 채널별 **활성화의 L2 Norm**을 측정합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`torch.norm(x, p=2, dim=1) ** 2`** *(제곱합 산출)*:
   - 전치된 입력 `x`의 형상이 `(hidden_size, batch_size)`이므로, 배치 축인 `dim=1`을 따라 L2 Norm을 계산해 제곱하면 모든 배치 샘플에 걸친 제곱합 $\sum_b x_{j,b}^2$ 를 `(hidden_size,)` 벡터로 얻습니다. 이를 배치마다 누적 평균한 뒤 최종 제곱근을 취하면 L2 Norm이 됩니다.

In [ ]:
# [코드]
def compute_activation_norm_sq(x: torch.Tensor) -> torch.Tensor:
    # x.shape: (hidden_size, batch_size)
    x = x.type(torch.float32)
    activation_norm = ### torch.norm(x, p=2, dim=1) ** 2 ### 작성 필요 ###  # 전치된 (hidden_size, batch) 활성화의 배치 축(dim=1) L2 Norm 제곱합 계산
    return activation_norm

## 핵심 개념 20. Wanda Pruning (Weights × Activations)

### 핵심 원리: Wanda (Pruning by Weights and Activations)
- **정의:** 각 가중치의 중요도를 $|W_{i,j}| \cdot \|X_j\|_2$ 로 평가하며, 특히 가중치 행렬 전체가 아니라 **각 출력 뉴런(행, `dim=1`) 단위로 비교하여 프루닝(Per-output comparison)**하는 것이 성능 비결입니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`round(col * sparsity)`**: 전체 원소가 아닌 **각 행의 열 개수(`col`)** 기준으로 제거 개수를 정합니다.
2. **`torch.abs(W) * input_feat[n]`**: `W` `(row, col)`과 1D 활성화 벡터 `input_feat[n]` `(col,)`을 곱하면 자동으로 열 방향으로 브로드캐스팅됩니다.
3. **`torch.kthvalue(importance, num_zeros_per_row, dim=1)[0]`**: `dim=1`(각 행 내부)별로 독립 임계값 `(row,)`을 구합니다.
4. **`importance > threshold.reshape(row, 1)`**: 행별 임계값을 `(row, 1)`로 2차원 정렬해야 `(row, col)` 행렬과 행 단위로 정확히 비교됩니다.

In [ ]:
# [코드]
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            row, col = W.shape
            num_zeros_per_row = ### round(col * sparsity) ### 작성 필요 ###  # 각 출력 뉴런(행)마다 제거할 입력 연결 개수 산출
            importance = ### torch.abs(W) * input_feat[n] ### 작성 필요 ###  # |가중치|와 입력 활성화 크기(input_feat[n])의 곱으로 실효 영향력 평가
            threshold = ### torch.kthvalue(importance, num_zeros_per_row, dim=1)[0] ### 작성 필요 ###  # 전체 행렬이 아닌 각 행 내부(dim=1)별로 독립적인 임계값 계산
            mask = ### importance > threshold.reshape(row, 1) ### 작성 필요 ###  # 행별 임계값을 (row, 1)로 정렬해 (row, col) 중요도 행렬과 행 단위 비교
            W.mul_(mask)

## 핵심 개념 21. Vanilla Speculative Decoding: Draft Generation & Target Verification

### 핵심 원리: Draft-then-Verify 패러다임 (손실 없는 LLM 가속)
- **정의:** 작고 빠른 Draft 모델로 먼저 `gamma`개의 토큰을 순차 추측(Drafting)한 뒤, 크고 정확한 Target 모델이 **단 1회의 Forward Pass로 `gamma+1`개 위치를 병렬 검증(Verification)**합니다. 메모리 대역폭 병목을 획기적으로 완화하면서도 출력 분포는 Target 모델과 100% 동일합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`greedy_step(model, x, cache)`** *(Draft 생성)*:
   - Draft 모델의 KV 캐시를 재사용하면서 `gamma`번 반복해 다음 후보 토큰들을 하나씩 생성하고 `tokens`에 모읍니다.
2. **`torch.cat([x, drafted], dim=-1)`** *(검증 시퀀스 조립)*:
   - 마지막 확정 토큰 `x` `(1, 1)` 뒤에 추측한 `drafted` `(1, gamma)`을 시퀀스 축(`dim=-1`)으로 이어 붙여야 Target 모델이 `[x -> d1, d1 -> d2, ..., d_gamma -> bonus]`를 한 번에 예측할 수 있습니다.
3. **`result.logits[0].argmax(-1)`** *(Target 모델 예측 추출)*:
   - 배치 차원(0)을 제거한 각 위치의 로짓에서 가장 높은 확률의 토큰 ID를 뽑아 Draft의 선택과 비교합니다.

In [ ]:
# [코드]
@torch.inference_mode()
def draft_tokens(model, x, cache, gamma):
    tokens = []
    for _ in range(gamma):
        x, cache = ### greedy_step(model, x, cache) ### 작성 필요 ###  # 작은 Draft 모델로 KV 캐시를 이어가며 gamma개 후보 토큰 순차 생성
        tokens.append(x)
    return torch.cat(tokens, dim=-1), cache

@torch.inference_mode()
def verify_draft(target, x, drafted, cache):
    verify_ids = ### torch.cat([x, drafted], dim=-1) ### 작성 필요 ###  # 마지막 확정 토큰 x와 gamma개 후보를 시퀀스 축(dim=-1)으로 연결
    result = target(verify_ids, past_key_values=cache, use_cache=True)
    preds = ### result.logits[0].argmax(-1) ### 작성 필요 ###  # Target 모델이 단 1회 Forward로 병렬 검증한 다음 토큰 argmax 예측 추출
    return preds, result.past_key_values

## 핵심 개념 22. EAGLE Tree Decoding: Attention Mask & Posterior Acceptance

### 핵심 원리: 트리 기반 병렬 드래프트 및 최장 경로 채택 (EAGLE)
- **정의:** 단일 체인이 아닌 트리 형태로 여러 후보 분기를 동시에 검증할 때, 각 트리 노드는 **루트부터 자기 자신의 부모 경로에 있는 노드만 참조(Attend)**해야 하며 다른 형제/분기 노드는 볼 수 없도록 `tree_mask`를 구성합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`tree_mask[child] |= tree_mask[parent]`** *(조상 마스크 상속)*:
   - 자기 자신(`eye`)과 루트(`0`)가 켜진 상태에서 `parent`가 볼 수 있던 모든 조상 마스크를 비트 OR(`|=`)로 누적 상속받으면, `child` 노드가 자신의 트리 조상 경로만 정확히 바라보게 됩니다.
2. **`accept_lengths.argmax()`** & **`accept_lengths[best_candidate]`** *(최장 채택 경로 선택)*:
   - 모든 트리 분기(`candidates`) 중 Target 모델과 연속으로 가장 길게 일치한(`cumprod`) 분기를 `best_candidate`로 고르고 그 채택 길이를 반환합니다.

In [ ]:
# [코드]
def build_tree_mask_and_positions(mask_index_list, total_tokens, device=None):
    tree_mask = torch.eye(total_tokens + 1, dtype=torch.bool, device=device)
    tree_mask[:, 0] = True
    for child, parent in enumerate(mask_index_list, start=1):
        ### tree_mask[child] |= tree_mask[parent] ### 작성 필요 ###
    tree_position_ids = tree_mask.sum(dim=1).long() - 1
    tree_mask = tree_mask.float()[None, None]
    return tree_mask, tree_position_ids

def select_best_branch(accept_lengths):
    best_candidate = ### accept_lengths.argmax() ### 작성 필요 ###  # 모든 트리 후보 분기 중 연속 일치 길이(accept_lengths)가 최장인 경로 선택
    accept_length = ### accept_lengths[best_candidate] ### 작성 필요 ###  # 선택된 최적 분기에서 확정 채택할 토큰 개수 반환
    return best_candidate, accept_length

## 핵심 개념 23. DFlash: Parallel Block Drafting

### 핵심 원리: Diffusion 기반 병렬 블록 드래프팅 (ICML'26 DFlash)
- **정의:** Draft 단계마저 순차(Auto-regressive) 루프로 돌지 않고, 경량 Block Diffusion 모델이 **단 1회의 Forward Pass로 블록 내 여러 미래 토큰을 한꺼번에 병렬 생성**하여 드래프팅 지연시간(Latency)을 극단적으로 줄입니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`draft_hidden[:, -num_draft:, :]`** *(제안 위치 슬라이싱)*:
   - 전체 블록 출력 `draft_hidden`에서 실제로 Target에 제안할 뒤쪽 `num_draft`(= `verify_size - 1`)개 토큰의 은닉 상태만 잘라냅니다.
2. **`model.compute_logits(draft_hidden, output_head)`** *(병렬 로짓 투영)*:
   - 추출된 `num_draft`개 위치의 은닉 벡터를 Target과 공유하는 `output_head`로 투영해 어휘집 로짓을 한 번에 계산합니다.
3. **`torch.argmax(draft_logits, dim=-1)`** *(동시 토큰 디코딩)*:
   - 마지막 어휘 차원(`dim=-1`)에서 `argmax`를 취해 `num_draft`개의 드래프트 토큰 ID를 루프 없이 단번에 얻습니다.

In [ ]:
# [코드]
def parallel_block_draft(model, target_hidden, noise_embedding, draft_position_ids,
                         past_key_values_draft, output_head, verify_size):
    draft_hidden = model(
        target_hidden=target_hidden,
        noise_embedding=noise_embedding,
        position_ids=draft_position_ids,
        past_key_values=past_key_values_draft,
        use_cache=True,
    )
    num_draft = verify_size - 1  # 검증 크기에서 기준 토큰 1개를 뺀 실제 제안 드래프트 토큰 개수
    draft_hidden = ### draft_hidden[:, -num_draft:, :] ### 작성 필요 ###  # 블록 출력 중 실제로 미래 토큰을 제안할 뒤쪽 num_draft개 은닉 상태 슬라이싱
    draft_logits = ### model.compute_logits(draft_hidden, output_head) ### 작성 필요 ###  # Target과 공유하는 output_head로 모든 제안 위치의 어휘 로짓을 병렬 계산
    draft_tokens = ### torch.argmax(draft_logits, dim=-1) ### 작성 필요 ###  # 루프 없이 어휘 축(dim=-1) argmax로 다수 드래프트 토큰을 한 번에 선택
    return draft_tokens